In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\DIGAMBARPATIL\AppData\Local\Temp\ipykernel_14812\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [3]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  X Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Memory Management_Stallings.pdf
  ✓ Loaded 45 pages

Processing: SPPU_IoT_Unit1_Detailed_Notes.pdf
  ✓ Loaded 12 pages

Total documents loaded: 57


In [4]:
all_pdf_documents


[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 'source': '..\\data\\pdf\\Memory Management_Stallings.pdf', 'total_pages': 45, 'page': 0, 'page_label': '1', 'source_file': 'Memory Management_Stallings.pdf', 'file_type': 'pdf'}, page_content='+ Chapter 4\nCache Memory\n© 2016 Pearson Education, Inc., Hoboken, NJ. All rights reserved.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 'source': '..\\data\\pdf\\Memory Management_Stallings.pdf', 'total_pages': 45, 'page': 1, 'page_label': '2', 'source_file': 'Memory Management_Stallings.pdf', 'file_type': 'pdf'}, page_content='Table 4.1    \nKey Characteristics of Computer Memory Systems \n© 2016 Pearson Education, Inc., Hoboken, NJ. All rights reserved.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 's

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 57 documents into 58 chunks

Example chunk:
Content: + Chapter 4
Cache Memory
© 2016 Pearson Education, Inc., Hoboken, NJ. All rights reserved....
Metadata: {'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 'source': '..\\data\\pdf\\Memory Management_Stallings.pdf', 'total_pages': 45, 'page': 0, 'page_label': '1', 'source_file': 'Memory Management_Stallings.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 'source': '..\\data\\pdf\\Memory Management_Stallings.pdf', 'total_pages': 45, 'page': 0, 'page_label': '1', 'source_file': 'Memory Management_Stallings.pdf', 'file_type': 'pdf'}, page_content='+ Chapter 4\nCache Memory\n© 2016 Pearson Education, Inc., Hoboken, NJ. All rights reserved.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 'source': '..\\data\\pdf\\Memory Management_Stallings.pdf', 'total_pages': 45, 'page': 1, 'page_label': '2', 'source_file': 'Memory Management_Stallings.pdf', 'file_type': 'pdf'}, page_content='Table 4.1    \nKey Characteristics of Computer Memory Systems \n© 2016 Pearson Education, Inc., Hoboken, NJ. All rights reserved.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 's

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
            
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

# --- MOVE THESE TO THE LEFT (OUTSIDE THE CLASS) ---
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\projects\new_mini_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DIGAMBARPATIL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully. Embedding dimension: 384


C:\Users\DIGAMBARPATIL\AppData\Local\Temp\ipykernel_14812\3332939220.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [9]:
import os
import uuid
import chromadb
import numpy as np
from typing import List, Any

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

# --- Corrected Execution ---
# Ensure this line is NOT indented inside the class
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 'source': '..\\data\\pdf\\Memory Management_Stallings.pdf', 'total_pages': 45, 'page': 0, 'page_label': '1', 'source_file': 'Memory Management_Stallings.pdf', 'file_type': 'pdf'}, page_content='+ Chapter 4\nCache Memory\n© 2016 Pearson Education, Inc., Hoboken, NJ. All rights reserved.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 'source': '..\\data\\pdf\\Memory Management_Stallings.pdf', 'total_pages': 45, 'page': 1, 'page_label': '2', 'source_file': 'Memory Management_Stallings.pdf', 'file_type': 'pdf'}, page_content='Table 4.1    \nKey Characteristics of Computer Memory Systems \n© 2016 Pearson Education, Inc., Hoboken, NJ. All rights reserved.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Memory Management_Stallings.pptx', 's

In [11]:
# 1. Extract the text
texts = [doc.page_content for doc in chunks]

# 2. Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# 3. Add to store
vectorstore.add_documents(chunks, embeddings)

print("✅ Success! Your PDF is now a searchable vector database.")

Generating embeddings for 58 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generated embeddings with shape: (58, 384)
Adding 58 documents to vector store...
Successfully added 58 documents to vector store
Total documents in collection: 58
✅ Success! Your PDF is now a searchable vector database.


In [12]:
from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        try:
            # Generate query embedding
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]

            # Search in vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

# --- Initialization ---
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("what are IoT Communication Models")

Retrieving documents for query: 'what are IoT Communication Models'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_2d89520c_48',
  'content': 'IoT Communication Models\n1. Device-to-Device (D2D)\n2. Device-to-Cloud\n3. Device-to-Gateway\n4. Back-End Data Sharing\nD2D enables direct communication.\nDevice-to-Cloud connects devices directly to cloud platforms.\nDevice-to-Gateway uses an intermediate gateway.\nBack-End Sharing allows cloud data sharing among applications.\nAdvantages include scalability, analytics, remote access, and integration.',
  'metadata': {'file_type': 'pdf',
   'moddate': '2026-06-12T15:50:16+00:00',
   'source_file': 'SPPU_IoT_Unit1_Detailed_Notes.pdf',
   'keywords': '',
   'page_label': '3',
   'creator': '(unspecified)',
   'content_length': 397,
   'page': 2,
   'creationdate': '2026-06-12T15:50:16+00:00',
   'author': '(anonymous)',
   'total_pages': 12,
   'doc_index': 48,
   'title': '(anonymous)',
   'source': '..\\data\\pdf\\SPPU_IoT_Unit1_Detailed_Notes.pdf',
   'subject': '(unspecified)',
   'trapped': '/False',
   'producer': 'ReportLab PDF Library - 

In [15]:
rag_retriever.retrieve("what are Replacement Algorithms")

Retrieving documents for query: 'what are Replacement Algorithms'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_cf5c556a_35',
  'content': '+ The most common replacement \nalgorithms are:\n■ Least recently used (LRU)\n■ Most effective\n■ Replace that block in the set that has been in the cache longest with \nno reference to it\n■ Because of its simplicity of implementation, LRU is the most popular \nreplacement algorithm\n■ First-in-first-out (FIFO)\n■ Replace that block in the set that has been in the cache longest\n■ Easily implemented as a round-robin or circular buffer technique\n■ Least frequently used (LFU)\n■ Replace that block in the set that has experienced the fewest \nreferences\n■ Could be implemented by associating a counter with each line\n© 2016 Pearson Education, Inc., Hoboken, NJ. All rights reserved.',
  'metadata': {'page': 34,
   'doc_index': 35,
   'source': '..\\data\\pdf\\Memory Management_Stallings.pdf',
   'producer': 'PyPDF',
   'file_type': 'pdf',
   'title': 'Memory Management_Stallings.pptx',
   'creationdate': '',
   'total_pages': 45,
   'page_label': 

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = "GROQ_API_KEY"

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024
)

def rag_simple(query, retriever, llm, top_k=3):
    # retrieve context
    results = retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [19]:
answer=rag_simple("what is replacement algorithms?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is replacement algorithms?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Replacement algorithms are methods used to determine which block in a cache to replace when a new block is brought in and the cache is full.


In [22]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score,
      and optionally full context.
    """

    results = retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=min_score
    )

    if not results:
        return {
            "answer": "No relevant context found.",
            "sources": [],
            "confidence": 0.0,
            "context": ""
        }

    # Prepare context
    context = "\n\n".join(
        [doc["content"] for doc in results]
    )

    # Prepare sources
    sources = [
        {
            "source": doc["metadata"].get(
                "source_file",
                doc["metadata"].get("source", "unknown")
            ),
            "page": doc["metadata"].get("page", "unknown"),
            "score": doc["similarity_score"],
            "preview": doc["content"][:300] + "..."
        }
        for doc in results
    ]

    confidence = max(
        [doc["similarity_score"] for doc in results]
    )

    # Generate answer
    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence": confidence
    }

    if return_context:
        output["context"] = context

    return output

result = rag_advanced(
    "What are Replacement Algorithms?",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

print("\nConfidence:")
print(result["confidence"])

print("\nContext Preview:")
print(result["context"][:300])

Retrieving documents for query: 'What are Replacement Algorithms?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Answer:
Replacement algorithms are methods used to determine which block in a cache to replace when it is full and a new block needs to be added, including LRU, FIFO, and LFU.

Sources:
[{'source': 'Memory Management_Stallings.pdf', 'page': 34, 'score': 0.21079999208450317, 'preview': '+ The most common replacement \nalgorithms are:\n■ Least recently used (LRU)\n■ Most effective\n■ Replace that block in the set that has been in the cache longest with \nno reference to it\n■ Because of its simplicity of implementation, LRU is the most popular \nreplacement algorithm\n■ First-in-first-out (...'}]

Confidence:
0.21079999208450317

Context Preview:
+ The most common replacement 
algorithms are:
■ Least recently used (LRU)
■ Most effective
■ Replace that block in the set that has been in the cache longest with 
no reference to it
■ Because of its simplicity of implementation, LRU is the most popular 
replaceme

In [30]:
from typing import List, Dict, Any
import time

# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---

class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # store query history

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:

        # Retrieve relevant documents
        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""

        else:
            context = "\n\n".join(
                [doc["content"] for doc in results]
            )

            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file",
                        doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get(
                        "page",
                        "unknown"
                    ),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:120] + "..."
                }
                for doc in results
            ]

            prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{question}

Answer:
"""

            # Streaming simulation
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end="", flush=True)
                    time.sleep(0.05)
                print()

            response = self.llm.invoke(prompt)
            answer = response.content

        # Add citations
        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]

        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations)
            if citations else answer
        )

        # Optionally summarize answer
        summary = None

        if summarize and answer:
            summary_prompt = (
                f"Summarize the following answer in 2 sentences:\n\n{answer}"
            )

            summary_resp = self.llm.invoke(summary_prompt)
            summary = summary_resp.content

        # Store query history
        self.history.append(
            {
                "question": question,
                "answer": answer,
                "sources": sources,
                "summary": summary
            }
        )

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)

result = adv_rag.query(
    "What are IoT Communication Models?",
    top_k=5,
    min_score=0.1,
    stream=True,
    summarize=True
)

print("\nFinal Answer:")
print(result["answer"])

print("\nSummary:")
print(result["summary"])

print("\nHistory:")
print(result["history"][-1])

Retrieving documents for query: 'What are IoT Communication Models?'
Top K: 5, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:

Use the following context to answer the question concisely.

Context:
IoT Communication Models
1. Device-to-Device (D2D)
2. Device-to-Cloud
3. Device-to-Gateway
4. Back-End Data Sharing
D2D enables direct communication.
Device-to-Cloud connects devices directly to cloud platforms.
Device-to-Gateway uses an intermediate gateway.
Back-End Sharing allows cloud data sharing among applications.
Advantages include scalability, analytics, remote access, and integration.

IoT Protocols (MQTT, CoAP, HTTP)
MQTT:
- Lightweight publish-subscribe protocol.
- Uses broker architecture.
- Low bandwidth consumption.
CoAP:
- Constrained Application Protocol.
- Uses UDP.
- Suitable for low-power devices.
HTTP:
- Request-response protocol.
- Widely used on the web.
- Easy integration with cloud services.
Comparison:
MQTT and CoAP are optimized for IoT, while HTTP is more general-purpose.

Question:
What ar